# Part 5 — 사전학습 네 방법의 병상 이탈 비교

기존 `rokey_ws/vision/bed_exit`의 **DetectionROI, SegmentationROI, YoloPoseROI,
MediaPipePoseROI**와 다운로드·평가 코드를 Part 5 패키지에 재사용했다. 학습은 수행하지 않는다.
모든 전략은 `BedExitMethod`를 상속하고 **BGR uint8 ndarray → judge(frame) → Python bool**을 제공한다.

출력 상태는 **SAFE=False / EXIT=True** 두 개다. 모델 미검출·가림·오류는 예외이며 정상으로 발행하지 않는다.
다수결은 기존 `BedExitState`만 사용한다: **초기 4프레임 False, 최근 5개 중 True ≥3**.
ROS는 유효 프레임마다 `/patient/bed_exit` (`std_msgs/msg/Bool`)을 발행한다.
평가에서는 별도 래치로 연속 경보를 한 이벤트로 세고, 유효 SAFE 원판정 5회 연속 후 재무장한다.

Figma 원본은 수정하지 않았다. 호출 한도로 현행 원본 대조는 못했으며 토픽/타입은 저장소 AGENTS.md 계약을 따른다.


**첫 실행:** 맨 위 의존성 점검·설치 코드 셀을 실행하세요. 설치 후 커널을 재시작하고 위에서부터 실행합니다. `RcParams._get` 오류가 났던 커널도 업데이트 후 재시작해야 합니다.


## 1. 환경

저장소에서 Python 3.10–3.12 가상환경을 만들고 이 노트북의 커널로 선택한다.
ROS Jazzy와 함께 쓸 때는 시스템 패키지를 볼 수 있는 가상환경을 사용한다.

```bash
python3 -m venv --system-site-packages .venv
source .venv/bin/activate
python -m pip install -r src/part5_bed_exit/requirements.txt
python -m ipykernel install --user --name part5-bed-exit
jupyter lab src/part5_bed_exit/notebooks/part5_bed_exit.ipynb
```

아래 경로 탐색은 저장소 루트 또는 노트북 폴더에서 실행할 수 있다.
모델 다운로드는 다음 절에서 실행한다. 데이터셋 다운로드는 제공자 접근 조건에 맞는 작업을 선택한다.


In [ ]:
# 0. 의존성 점검·설치 (처음 한 번). 설치 후에는 커널을 재시작하세요.
import importlib.metadata as _metadata
from pathlib import Path as _Path
import subprocess as _subprocess
import sys as _sys
from packaging.requirements import Requirement as _Requirement

_roots = [_Path.cwd(), *_Path.cwd().parents]
_requirements = next((candidate for base in _roots
                      for candidate in [base / 'src' / 'part5_bed_exit' / 'requirements.txt',
                                        base / 'requirements.txt']
                      if candidate.is_file() and candidate.parent.name == 'part5_bed_exit'), None)
if _requirements is None:
    raise RuntimeError('이 저장소 또는 Part 5 폴더에서 노트북을 실행하세요.')
_needed = []
for _line in _requirements.read_text().splitlines():
    _line = _line.split('#', 1)[0].strip()
    if not _line:
        continue
    _requirement = _Requirement(_line)
    try:
        _version = _metadata.version(_requirement.name)
    except _metadata.PackageNotFoundError:
        _needed.append(_line)
        continue
    if not _requirement.specifier.contains(_version):
        _needed.append(_line)
print('현재 커널 Python:', _sys.executable)
if _needed:
    print('설치/업데이트할 의존성:', ', '.join(_needed))
    _subprocess.check_call([_sys.executable, '-m', 'pip', 'install', '--upgrade',
                           '-r', str(_requirements)])
    raise RuntimeError('설치 완료. Kernel → Restart Kernel로 재시작한 뒤 이 셀부터 다시 실행하세요.')
if 'matplotlib' in _sys.modules and not hasattr(_sys.modules['matplotlib'].rcParams, '_get'):
    raise RuntimeError('이 커널에 이전 Matplotlib이 남아 있습니다. Kernel → Restart Kernel 후 다시 실행하세요.')
print('의존성 준비 완료. 다음 셀부터 실행하세요.')
print('Matplotlib:', _metadata.version('matplotlib'),
      '| matplotlib-inline:', _metadata.version('matplotlib-inline'))


In [ ]:
import json
import os
from pathlib import Path
import shutil
import sys

search = [Path.cwd(), *Path.cwd().parents]
PACKAGE_ROOT = next((candidate for root in search
                     for candidate in [root / 'src' / 'part5_bed_exit', root]
                     if (candidate / 'part5_bed_exit' / 'bed_exit_method.py').is_file()), None)
if PACKAGE_ROOT is None:
    raise RuntimeError('이 저장소 또는 Part 5 폴더에서 노트북을 실행하세요.')
sys.path.insert(0, str(PACKAGE_ROOT))

from IPython.display import Markdown, display
import cv2
import numpy as np
import matplotlib.pyplot as plt
from part5_bed_exit.bed_exit_method import BedExitMethod
from part5_bed_exit.bed_exit_state import BedExitState
from part5_bed_exit.methods.bed_exit_core import (
    MODEL_FILES, BedExitMonitor, JudgmentUnavailable, draw_overlay,
)
from part5_bed_exit.pretrained import create_method
from part5_bed_exit.downloads import (
    download_models, download_roboflow, download_slp_samples,
    fallvision_inventory, download_fallvision_files, extract_zip, sha256,
)
from part5_bed_exit.evaluation import video_frames, run_benchmark
from part5_bed_exit.input_video import (
    InputVideoUnavailable, first_video_frame, prepare_input_video,
)

MODEL_DIR = Path(os.environ.get('BED_EXIT_MODEL_DIR', str(PACKAGE_ROOT / 'models'))).expanduser().resolve()
DATA_DIR = PACKAGE_ROOT / 'data'
RESULTS_DIR = PACKAGE_ROOT / 'results'
METHOD_NAMES = list(MODEL_FILES)
print('Part 5:', PACKAGE_ROOT)
print('Models:', MODEL_DIR)


## 2. 사전학습 모델 다운로드 / 기존 가중치 재사용

YOLO11 세 가중치와 MediaPipe Full task를 공식 배포처에서 받는다(모델당 최대 50 MiB).
`.source.json`의 SHA256 영수증이 일치하면 다시 받지 않는다.
이미 확보한 모델 폴더가 있으면 위 `MODEL_DIR`을 그 폴더로 지정하고 `DOWNLOAD_MODELS=False`로 둔다.
가중치는 Git에 포함하지 않는다. 모델/코드 라이선스는 공식 배포 문서를 따른다.


In [11]:
DOWNLOAD_MODELS = os.environ.get('BED_EXIT_DOWNLOAD_MODELS', '1') == '1'
if DOWNLOAD_MODELS:
    for path in download_models(MODEL_DIR):
        print(path.name, path.stat().st_size, 'bytes', sha256(path))
else:
    print('모델 다운로드 생략. 기존 MODEL_DIR을 사용합니다.')
print({name: (MODEL_DIR / filename).is_file() for name, filename in MODEL_FILES.items()})


yolo11n.pt 5613764 bytes 0ebbc80d4a7680d14987a577cd21342b65ecfd94632bd9a8da63ae6417644ee1
yolo11n-seg.pt 6182636 bytes 55ed65c56c91713d23e8402371c6c49a6fd84f257f7dce452e8d70e41dcbe152
yolo11n-pose.pt 6255593 bytes 869e83fcdffdc7371fa4e34cd8e51c838cc729571d1635e5141e3075e9319dc0
pose_landmarker_full.task 9398198 bytes 5134a3aad27a58b93da0088d431f366da362b44e3ccfbe3462b3827a839011b1
{'detection': True, 'segmentation': True, 'yolo_pose': True, 'mediapipe_pose': True}


## 3. 데이터셋 조사와 다운로드

`dataset_catalog.json`은 기존 조사 결과의 출처·접근 조건을 보존한다. 접근 실패와 다운로드 성공을 구별한다.

| 자료 | 용도 | 다운로드 경로 / 한계 |
|---|---|---|
| Roboflow Bed Exit v4 | 정적 RGB와 박스, ROI 점검 | `roboflow` 설치 + `ROBOFLOW_API_KEY`; 이벤트 시간 GT 없음 |
| FallVision | 원본 RGB 영상, 이탈·낙상 움직임 | inventory 확인 후 파일 ID 선택; API 차단 시 공식 Files 탭에서 수동 다운로드 |
| SLP2022 | 누운 자세·이불 가림의 정적 RGB | Range로 조건별 소량 추출; 제공자 비밀번호 `SLP_ZIP_PASSWORD` 필요 |
| BlanketSet | 이불·정상 움직임 오경보 | 현재 조사상 영상 접근 제한; 제공자 승인 파일 사용 |
| In-Bed Patient Segmentation | person/bed 마스크 | 저자 요청, 공개 ZIP 미확인 |
| TIP/PIMesh | 누운 자세 관절 평가 | 제공자 요청 절차, 원본 접근 확인 필요 |

정적 사진을 복제해 만든 영상은 Lead Time 평가에 쓰지 않는다. 낙상 클립 레이블도 이탈 시각 정답과 다르다.
정량평가는 원본 RGB와 수동 검수한 Bed/Safe ROI, 위험 시작·경계 이탈·종료 시각이 필요하다.
토큰/비밀번호는 커널을 실행하기 전 환경변수로 설정하며 노트북에 저장하지 않는다.


기본 `auto_video` 모드는 이미 받은 원본 영상을 자동 탐색하고, 없으면 FallVision 원본 다운로드를 시도합니다. 모델 4개 다운로드와 입력 영상 다운로드는 별도입니다. 실제 영상 연결 성공/실패와 선택 경로를 바로 출력합니다. `BED_EXIT_DATA_DIR`을 지정하면 다른 데이터셋 다운로드 폴더도 자동 탐색합니다.


In [ ]:
catalog = json.loads((PACKAGE_ROOT / 'dataset_catalog.json').read_text(encoding='utf-8'))
for dataset in catalog['datasets']:
    print(dataset['name'], '→', dataset['url'])

# 기본값: 실제 다운로드된 원본 영상 자동 연결; 없으면 원본 다운로드 시도
# auto_video / catalog / roboflow / slp / fallvision_inventory / fallvision_download
DATASET_ACTION = 'auto_video'
FALLVISION_FILE_IDS = []  # inventory의 실제 ID를 선택. 예시 ID를 임의로 넣지 않는다.
MAX_DATA_BYTES = 300 * 1024**2

VIDEO_PATH = None
if DATASET_ACTION == 'auto_video':
    print('다운로드 폴더에서 원본 RGB 영상을 연결합니다. 없으면 FallVision 다운로드를 시도합니다.')
    try:
        selected_video = prepare_input_video(DATA_DIR, video_path=os.environ.get('BED_EXIT_VIDEO'))
        VIDEO_PATH = str(selected_video)
        print('입력 영상 자동 연결 완료:', VIDEO_PATH)
    except InputVideoUnavailable as exc:
        print(str(exc))
elif DATASET_ACTION == 'roboflow':
    # 필요 시 별도 셀에서 %pip install roboflow
    downloaded = download_roboflow(DATA_DIR / 'roboflow_bed_exit_v4')
    print('다운로드 경로:', downloaded.location)
elif DATASET_ACTION == 'slp':
    records = download_slp_samples(DATA_DIR / 'slp', count_per_cover=2)
    print('실제 추출한 정적 RGB:', len(records))
elif DATASET_ACTION == 'fallvision_inventory':
    inventory = fallvision_inventory(DATA_DIR / 'fallvision')
    display(inventory)  # 이름·크기·restricted·checksum을 검수한다.
elif DATASET_ACTION == 'fallvision_download':
    inventory = json.loads((DATA_DIR / 'fallvision' / 'inventory.json').read_text())['files']
    downloaded = download_fallvision_files(inventory, FALLVISION_FILE_IDS,
                                          DATA_DIR / 'fallvision', max_bytes=MAX_DATA_BYTES)
    for path in downloaded:
        print('다운로드 완료:', path.name)
        if path.suffix.lower() == '.zip':
            videos = extract_zip(path, DATA_DIR / 'fallvision' / 'extracted' / path.stem)
            print('추출한 MP4:', len(videos))
elif DATASET_ACTION != 'catalog':
    raise ValueError('알 수 없는 DATASET_ACTION')

if DATASET_ACTION != 'auto_video':
    selected_video = prepare_input_video(DATA_DIR, video_path=os.environ.get('BED_EXIT_VIDEO'), download=False)
    VIDEO_PATH = str(selected_video) if selected_video else None
    if VIDEO_PATH:
        print('입력 영상 자동 연결 완료:', VIDEO_PATH)


## 4. 카메라 ROI와 판정 정책 설정

두 JSON 로컬 사본을 만든다. **예시 좌표와 정답을 검증된 데이터로 취급하지 않는다.**
`methods.local.json`은 웹캠/미리보기/ROS용, `testbed.local.json`은 영상별 평가용이다.
정규화 좌표는 왼쪽 위 `(0,0)`, 오른쪽 아래 `(1,1)`이다.
실제 화면에서 Bed ROI와 그 내부 Safe ROI를 그린 뒤 `roi_version`을 정하고 `roi_reviewed=true`로 바꾼다.

기존 네 모듈의 기본 `any_body` 정책을 재사용한다. Detection은 박스 외부 비율,
Segmentation은 마스크 외부 비율, Pose 두 방법은 공통 관절의 외부 여부를 사용한다.
이 세 표현은 해부학적으로 같지 않다. 팔만 나온 경우와 몸통 이탈을 구분해 비교한다.
`outside_fraction=0.05`, 관절 신뢰도 0.5, 허용 오차 3px는 **미검증 초기값**이다.
`torso` 포즈 정책은 별도 실험용이며 정식 any_body 비교에 섞지 않는다.


In [ ]:
CONFIG_PATH = Path(os.environ.get('PART5_BED_EXIT_CONFIG', str(PACKAGE_ROOT / 'examples' / 'methods.local.json')))
TESTBED_PATH = PACKAGE_ROOT / 'examples' / 'testbed.local.json'
for template, destination in [('methods.template.json', CONFIG_PATH), ('testbed.template.json', TESTBED_PATH)]:
    if not destination.exists():
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(PACKAGE_ROOT / 'examples' / template, destination)
        if destination == CONFIG_PATH:
            config = json.loads(destination.read_text())
            config['model_dir'] = str(MODEL_DIR)
            destination.write_text(json.dumps(config, indent=2, ensure_ascii=False), encoding='utf-8')
print('판정 설정:', CONFIG_PATH)
print('평가 정답:', TESTBED_PATH)

# 위 데이터셋 셀에서 연결한 경로를 그대로 사용합니다.
if not VIDEO_PATH:
    selected_video = prepare_input_video(DATA_DIR, video_path=os.environ.get('BED_EXIT_VIDEO'), download=False)
    VIDEO_PATH = str(selected_video) if selected_video else None
if VIDEO_PATH:
    first_frame = first_video_frame(VIDEO_PATH)
    print('연결한 입력 영상:', VIDEO_PATH)
    config = json.loads(CONFIG_PATH.read_text())
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.imshow(cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB), extent=(0,1,1,0))
    for key, color in [('bed_roi', 'orange'), ('safe_roi', 'lime')]:
        points = np.array(config[key] + config[key][:1])
        ax.plot(points[:,0], points[:,1], color=color, label=key)
    ax.legend(); ax.set_title('Review normalized ROI on the actual camera view')
    plt.show()
else:
    print('입력 영상 다운로드가 아직 완료되지 않았습니다. 위 데이터셋 셀의 연결 결과를 확인하세요. ' '모델 파일만으로는 영상 추론이 시작되지 않습니다.')


## 5. 동일 프레임에서 네 메서드 연결 확인

`create_method()`는 ROS 노드와 같은 팩터리다. 각 모델은 한 번 초기화하고 이후 프레임에서 재사용한다.
여기서는 첫 프레임 API 연결만 확인하며 병상 이탈 성능으로 보고하지 않는다.
메서드의 원판정은 다수결 전 bool이다. 미검출·누락 관절은 `JudgmentUnavailable` 예외다.


In [ ]:
if VIDEO_PATH and json.loads(CONFIG_PATH.read_text()).get('roi_reviewed') is True:
    sample_frame = first_video_frame(VIDEO_PATH)
    for name in METHOD_NAMES:
        detector = create_method(name, CONFIG_PATH)
        try:
            assert isinstance(detector, BedExitMethod)
            try:
                raw = detector.judge(sample_frame)
                assert type(raw) is bool
                print(name, 'raw=', raw, detector.last.reason)
            except JudgmentUnavailable as exc:
                print(name, '판정 불가 / 미발행:', exc)
        finally:
            detector.close()
else:
    print('영상이 연결되면 위 셀에 첫 프레임이 표시됩니다. 침대 ROI 검수 후 roi_reviewed=true로 설정하세요.')

# 기존 다수결 규약의 간단한 예: 모델 추론 성능과는 별개인 논리 예제다.
state = BedExitState()
assert [state.update(True) for _ in range(5)] == [False, False, False, False, True]


## 6. 로컬 영상 / 웹캠 미리보기

한 메서드를 선택한다. 오류 프레임은 화면에 `No decision (not published)`로 표시하고 이력을 초기화한다.
미리보기는 최대 300프레임이며 `q` 또는 ESC로 종료한다. 창이 없는 환경에서는 실행하지 않는다.
ROS에는 `BedExitMonitor`가 아니라 `create_method()`로 얻는 원판정 객체를 연결해 이중 투표를 피한다.


In [14]:
def preview(method_name, source, max_frames=300):
    detector = create_method(method_name, CONFIG_PATH)
    monitor = BedExitMonitor(detector)
    capture = None
    try:
        capture = cv2.VideoCapture(source)
        if not capture.isOpened():
            raise RuntimeError('입력 카메라/영상을 열 수 없습니다.')
        for index in range(max_frames):
            try:
                ok, frame = capture.read()
                if not ok or frame is None:
                    monitor.gate.update(None, valid=False)
                    print('입력 종료/수신 실패: 추가 판정 없음')
                    break
            except Exception:
                monitor.gate.update(None, valid=False)
                raise
            try:
                voted = monitor.judge(frame)
                if monitor.event:
                    print('EXIT event at processed frame', index, 'vote=', voted)
            except Exception as exc:
                print('판정 미발행:', type(exc).__name__, str(exc))
            cv2.imshow('Part 5 bed exit', draw_overlay(frame, detector, monitor.result))
            if cv2.waitKey(1) & 0xff in (ord('q'), 27):
                break
    finally:
        if capture is not None:
            capture.release()
        monitor.close()
        cv2.destroyAllWindows()

RUN_PREVIEW = False
SELECTED_METHOD = 'yolo_pose'
PREVIEW_SOURCE = VIDEO_PATH if VIDEO_PATH else 0
if RUN_PREVIEW:
    preview(SELECTED_METHOD, PREVIEW_SOURCE)


## 7. 공통 테스트베드 정답 준비

`testbed.local.json`의 각 클립에 다음을 기록한다. 영상 경로는 **JSON 파일 기준 상대 경로** 또는 절대 경로다.

- 원본 파일 SHA256, 영상 길이, 가명 `subject_id/session_id`, `validation/test` split.
- 실제 Bed/Safe ROI와 버전, 검수 후 `labels_reviewed/roi_reviewed=true`.
- 사건별 `risk_start_s ≤ boundary_cross_s < end_s`, 고유 event_id. 정답이 없는 정상 클립은 검수 후 `events=[]`.
- 환자가 실제 보이는 시간 구간 `patient_present_intervals_s`; C01–C20, 이불·조명·침대 이동·도메인 조건.

참가자와 세션은 validation/test를 가로질러 섞지 않는다. 임계값은 validation에서만 결정한다.
기존 정책에 맞춰 허가 여부와 무관한 **기하학적 이탈**을 평가하고 허가 정보는 조건으로 남긴다.
의료진의 이탈 허가 판단은 이 영상 모델의 기능이 아니다.

아래 도구는 해시와 타임스탬프를 확인할 뿐 정답 이벤트나 ROI를 자동 생성하지 않는다.


In [ ]:
def inspect_video(path):
    import av
    path = Path(path).expanduser().resolve()
    with av.open(str(path)) as container:
        stream = container.streams.video[0]
        duration = float(stream.duration * stream.time_base) if stream.duration is not None else None
    frames = list((i,t) for i,t,_ in video_frames(path))
    return {'video': str(path), 'video_sha256': sha256(path), 'duration_s': duration,
            'frames': len(frames), 'first_pts_s': frames[0][1] if frames else None,
            'last_pts_s': frames[-1][1] if frames else None}

if VIDEO_PATH:
    import importlib.util
    if importlib.util.find_spec('av') is not None:
        display(inspect_video(VIDEO_PATH))
    else:
        print('영상 미리보기는 가능합니다. PTS 정량평가에는 현재 커널에서 %pip install av 가 필요합니다.')


## 8. 이벤트 정량평가 — 네 방법 독립 순차 실행

동일한 원본 프레임·PTS·ROI·정답·장비를 사용한다. 네 모델을 동시에 GPU에서 돌리지 않는다.
모델 로딩·첫 3회 warm-up은 속도 지표에서 제외한다. warm-up 후 환자 선택·다수결을 초기화한다.

| 지표 | 집계 |
|---|---|
| Event Precision / Recall | 경보와 GT를 1:1 매칭, 중복 경보는 FP, 미검출은 FN |
| Early Warning Recall | 실제 경계 이탈보다 먼저 경고한 GT / 전체 GT |
| FA/10m, FA/h | FP / 전체 영상 시간 × 600 또는 3600; 정상 구간 분모도 별도 제공 |
| Warning Lead Time | 실제 경계 이탈 − 경고 이용 가능 시각; 양수일 때만 사전 경고 |
| Detection Delay P95 | 위험 시작 → 경고 이용 가능 시각; 매칭된 사건 기준이며 FN도 함께 확인 |
| FPS / 추론 P95 | 전체 디코드·판정 처리량 / 동기화한 judge 호출 시간 |
| Tracking Lost Rate | 환자가 존재하는 시간 중 판단 불가 시간, 원본 PTS로 가중 |

정확도용 **signal_only**와 **FIFO 처리시간을 누적한 경고 이용 가능 시각**을 따로 기록한다.
FIFO는 모든 원본 프레임을 순차 처리하는 시뮬레이션이며, 프레임을 놓칠 수 있는 ROS 웹캠 실시간 지연을 실측한 값이 아니다.
오류도 CSV에 빈 판정·오류 사유로 기록한다. 모델 오류로 계산이 빨라지는 경우 invalid_frames를 함께 본다.

Notion의 수치는 제안 목표다: Precision ≥90%, FA/10m ≤0.25, Lead Time 0.3s,
Delay P95 ≤0.1s, ≥5 FPS, 추론 P95 ≤200ms, Early Recall ≥95%.
보고서는 Lead Time 목표에 중앙값을 적용하고 P05도 제공한다. 이는 모든 사건의 0.3초 보장을 뜻하지 않는다.
**5 FPS로 시작하면 5프레임 수집만 약 0.8초**가 필요하므로 지연 0.1초 목표와 동시에 달성하기 어렵다.
관측이 없거나 분모가 0인 지표는 0점이나 통과로 바꾸지 않고 `null`로 둔다.


In [8]:
RUN_BENCHMARK = False  # 실제 영상·정답 검수 후 True
EVALUATION_SPLIT = 'test'
comparison = None
if RUN_BENCHMARK:
    from datetime import datetime, timezone
    run_dir = RESULTS_DIR / datetime.now(timezone.utc).strftime('benchmark_%Y%m%dT%H%M%S_%fZ')
    comparison = run_benchmark(TESTBED_PATH, MODEL_DIR, run_dir,
                               methods=METHOD_NAMES, device='cpu', split=EVALUATION_SPLIT)
    columns = ['method', 'event_precision', 'event_recall', 'early_warning_recall',
               'false_alarms_per_10m', 'warning_lead_median_s', 'detection_delay_p95_s',
               'fps', 'judge_latency_p95_ms', 'tracking_lost_rate', 'invalid_frames']
    display([{key: row.get(key) for key in columns} for row in comparison])
    print('CSV / JSON / 조건별 리포트:', run_dir)
else:
    print('정답 미검수 상태: 정량평가를 실행하지 않았습니다.')


정답 미검수 상태: 정량평가를 실행하지 않았습니다.


In [9]:
if comparison:
    fields = [('event_recall','Event Recall'), ('false_alarms_per_10m','False Alarms / 10m'),
              ('warning_lead_median_s','Median Lead Time (s)'), ('detection_delay_p95_s','P95 Delay (s)'),
              ('fps','Throughput FPS'), ('tracking_lost_rate','Tracking Lost Rate')]
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for ax, (key, title) in zip(axes.flat, fields):
        values = [r.get(key) if r.get(key) is not None else np.nan for r in comparison]
        ax.bar([r['method'] for r in comparison], values)
        ax.set_title(title)
        ax.tick_params(axis='x', rotation=30)
    fig.tight_layout()
    fig.savefig(run_dir / 'comparison.png', dpi=180)
    plt.show()


## 9. ROS 2 Jazzy 연결

저장소 루트에서 아래처럼 실행한다. `method` 값만 네 이름 중 하나로 바꾸면 같은 설정을 사용한다.
설정 파일과 모델 경로는 실행하는 컴퓨터에 존재해야 한다.

```bash
source /opt/ros/jazzy/setup.bash
colcon build --packages-select part5_bed_exit --symlink-install
source install/setup.bash
ros2 run part5_bed_exit bed_exit --ros-args \
  -p method:=yolo_pose \
  -p method_config:=$(pwd)/src/part5_bed_exit/examples/methods.local.json \
  -p output_topic:=/patient/bed_exit \
  -p camera_index:=0 \
  -p capture_fps:=30.0
```

`capture_fps`는 수집 시도 주기다. 카메라와 순차 메서드 실행 시간에 제한되며 모든 센서 프레임 처리를 보장하지 않는다.
외부 클래스/객체 `module:ClassName` 연결도 유지된다. 기존 객체는 자체 `judge(frame)`을 제공하면 된다.


## 근거와 재사용 출처

- [요구사항·공통 테스트베드 Mermaid](https://app.notion.com/p/419e8369193381589056f77090c43086)
- [방법론 A–D와 평가 기준](https://app.notion.com/p/418e836919338100aa5ffb0eb12345cb)
- [데이터셋 조사](https://app.notion.com/p/419e8369193381018cbad16a32a4af98)
- [Figma Part 05 원본, 읽기 전용](https://www.figma.com/board/30YLtBcO7afOCGHlrgocjM/?node-id=0-1)
- [Ultralytics YOLO11](https://docs.ultralytics.com/models/yolo11/), [NumPy BGR 입력·Results API](https://docs.ultralytics.com/modes/predict/)
- [MediaPipe Pose Landmarker](https://ai.google.dev/edge/mediapipe/solutions/vision/pose_landmarker/python)

재사용 원본: `/home/hv-02/rokey_ws/vision/bed_exit/{bed_exit_core.py,downloads.py,evaluation.py,dataset_catalog.json}`.
원본 폴더는 변경하지 않았다. 통합 시 판단 불가 bool 보류 동작을 예외로 바꾸고,
이벤트 게이트를 저장소 `BedExitState`에 연결해 초기 4프레임·3/5 계약을 일치시켰다.
